# Milestone 19 - Performance Testing Agent

This notebook traces the standalone Performance Testing Agent. The agent reads `performance_tests` from `test_plan` or infers a safe GET/HEAD endpoint, runs a small Locust-based load test, and saves `performance_result.json`.

## Role and Safety

The Performance Agent works alone before main workflow integration. It uses small load settings by default, tests localhost targets only unless explicitly allowed, does not start Django, does not execute repository code, and does not call LLM APIs.

## Mini Workflow

```text
START -> performance_testing -> END
```

Reads from State: `target_url`, `test_plan`, `user_preferences`, `discovered_endpoints`.

Writes to State: `performance_results`, `performance_result_path`, `performance_artifacts`, `errors`, `agent_logs`.

Target repository: https://github.com/Vitaee/DjangoRestAPI

Target URL: http://localhost:8000

## Part A - Unit-style execution with mocked Locust

In [ ]:
from test_auto.agents import performance_testing_agent
from test_auto.agents.performance_testing_agent import run_performance_testing_agent_alone

fake_test_plan = {
    "performance_tests": [
        {
            "id": "PERF_001",
            "name": "todo_list_perf",
            "endpoint": "/api/todos/",
            "method": "GET",
            "users": 1,
            "spawn_rate": 1,
            "duration_seconds": 1,
        }
    ]
}

def fake_execute(target_url, test_case, run_id, user_preferences=None):
    return {
        "id": test_case.get("id", "PERF_001"),
        "name": test_case.get("name", "todo_list_perf"),
        "endpoint": test_case.get("endpoint", "/api/todos/"),
        "method": "GET",
        "status": "passed",
        "users": 1,
        "spawn_rate": 1.0,
        "duration_seconds": 1,
        "total_requests": 10,
        "failures": 0,
        "failure_rate": 0.0,
        "average_response_time_ms": 25.0,
        "min_response_time_ms": 10.0,
        "max_response_time_ms": 50.0,
        "p50_response_time_ms": 20.0,
        "p95_response_time_ms": 45.0,
        "requests_per_second": 5.0,
        "threshold_results": [{"name": "average_response_time", "passed": True}],
        "details": "mocked notebook execution",
        "error_type": None,
        "artifact_paths": [f"results/runs/{run_id}/performance/locustfile_PERF_001.py"],
    }

original_execute = performance_testing_agent.execute_performance_test_case
performance_testing_agent.execute_performance_test_case = fake_execute
try:
    result = run_performance_testing_agent_alone(
        target_url="http://localhost:8000",
        test_plan=fake_test_plan,
        run_id="notebook_perf_mocked",
    )
finally:
    performance_testing_agent.execute_performance_test_case = original_execute

result["summary"], result["performance_result_path"]

## Part B - Optional real localhost target

This requires the Django target app running at `http://localhost:8000` and Locust installed in the selected kernel. Keep `RUN_REAL_PERFORMANCE = False` unless the local app is ready.

In [ ]:
RUN_REAL_PERFORMANCE = False

if RUN_REAL_PERFORMANCE:
    real_result = run_performance_testing_agent_alone(
        target_url="http://localhost:8000",
        test_plan=fake_test_plan,
        run_id="notebook_perf_real",
    )
    print(real_result["summary"])
    print(real_result["performance_result_path"])
else:
    print("Skipped real performance run. Start the local target app first, then set RUN_REAL_PERFORMANCE = True.")

## Part C - Mini LangGraph workflow

In [ ]:
from test_auto.graph.performance_testing_workflow import run_performance_testing_workflow

original_execute = performance_testing_agent.execute_performance_test_case
performance_testing_agent.execute_performance_test_case = fake_execute
try:
    final_state = run_performance_testing_workflow(
        {
            "run_id": "notebook_perf_workflow",
            "target_url": "http://localhost:8000",
            "test_plan": fake_test_plan,
            "errors": [],
            "agent_logs": [],
        }
    )
finally:
    performance_testing_agent.execute_performance_test_case = original_execute

final_state["performance_results"]["summary"]